<a href="https://colab.research.google.com/github/davis-mironga/marsabit-ecosystem-analysis/blob/main/02_LULC_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📓 Notebook 2 — LULC Classification
## The Marsabit Footprint: Random Forest Land Cover Classification (1990–2024)

---

**Purpose of this Notebook:**
Using the NDVI composites exported in Notebook 1, we train a Random Forest
machine learning model to classify every pixel in Marsabit County into one
of 5 land cover types — for each of our 5 time periods.

**What this notebook produces:**
- ✅ Trained Random Forest classifier (per time period)
- ✅ Land cover maps for 1990, 2000, 2010, 2020, 2024
- ✅ Accuracy reports (Overall Accuracy, Kappa, Confusion Matrix)
- ✅ Change detection matrix (e.g. Forest → Bareland transitions)
- ✅ Area statistics per class per decade (hectares)
- ✅ All classified maps exported to GEE Assets

**The 5 Land Cover Classes:**
| Class | Code | Description |
|-------|------|-------------|
| Forest | 1 | Dense closed canopy — Mt. Marsabit & Mt. Kulal |
| Rangeland | 2 | Open grassland and shrubland — mid-elevation plains |
| Wetlands | 3 | Seasonally/permanently wet — riparian zones & pans |
| River | 4 | Permanent river channels — Milgis, Uaso Nyiro |
| Bareland | 5 | Exposed soil, rock, desert margins |

**Notebooks in this project:**
| Notebook | Purpose | Status |
|----------|---------|--------|
| 01_GEE_Data_Preprocessing | Satellite data, NDVI, exports | ✅ Complete |
| **02_LULC_Classification** ← You are here | Random Forest land cover maps | 🔄 In Progress |
| 03_Spatial_Analysis | Moran's I, LISA hotspots, LCI | ⏳ Pending |
| 04_Regression_Modeling | OLS, GWR, vulnerability map | ⏳ Pending |

---

## ⚙️ STEP 1 — Install Libraries, Authenticate & Load GEE Assets

**What we are doing:**
We reinstall geemap, re-authenticate GEE, and immediately load all 13 assets
that were exported in Notebook 1. Loading from GEE Assets means we do NOT
need to reprocess any satellite imagery — all the heavy lifting was already
done in Notebook 1.

**Why we reload everything:**
Colab is a temporary environment — every new session loses all variables.
By loading from GEE Assets at the start, Notebook 2 is fully independent
and can run on its own without re-running Notebook 1.

> ⚠️ When ee.Authenticate() runs, log in with the same Google account
> used in Notebook 1 (davismironga@gmail.com).

In [4]:
# Install geemap (required every new Colab session)
!pip install geemap -q

import ee
import geemap
import math

# Authenticate and initialize GEE
ee.Authenticate()
ee.Initialize(project='mironga-project-marsabit')

# ─────────────────────────────────────────────────────────────────────
# Load all 13 assets exported from Notebook 1
# These are permanently stored in GEE — no reprocessing needed
# ─────────────────────────────────────────────────────────────────────
ASSET_PATH = 'projects/mironga-project-marsabit/assets/marsabit'

# NDVI Composites
ndvi_1990 = ee.Image(f'{ASSET_PATH}/ndvi_1990')
ndvi_2000 = ee.Image(f'{ASSET_PATH}/ndvi_2000')
ndvi_2010 = ee.Image(f'{ASSET_PATH}/ndvi_2010')
ndvi_2020 = ee.Image(f'{ASSET_PATH}/ndvi_2020')
ndvi_2024 = ee.Image(f'{ASSET_PATH}/ndvi_2024')

# NDVI Change Maps
ndvi_change_90_00 = ee.Image(f'{ASSET_PATH}/ndvi_change_1990_2000')
ndvi_change_00_10 = ee.Image(f'{ASSET_PATH}/ndvi_change_2000_2010')
ndvi_change_10_20 = ee.Image(f'{ASSET_PATH}/ndvi_change_2010_2020')
ndvi_change_20_24 = ee.Image(f'{ASSET_PATH}/ndvi_change_2020_2024')
ndvi_change_total = ee.Image(f'{ASSET_PATH}/ndvi_change_total')

# Control Variables
rainfall_change = ee.Image(f'{ASSET_PATH}/rainfall_change')
elevation       = ee.Image(f'{ASSET_PATH}/elevation')
lci_distance    = ee.Image(f'{ASSET_PATH}/lci_distance')

# Marsabit boundary
marsabit_roi  = (ee.FeatureCollection("FAO/GAUL/2015/level2")
                .filter(ee.Filter.eq('ADM2_NAME', 'Marsabit')))
marsabit_geom = marsabit_roi.geometry()

print("✅ GEE authenticated and initialized!")
print(f"\n📦 Assets loaded from: {ASSET_PATH}/")
print("   ✓ ndvi_1990, ndvi_2000, ndvi_2010, ndvi_2020, ndvi_2024")
print("   ✓ ndvi_change layers (4 decadal + 1 total)")
print("   ✓ rainfall_change, elevation, lci_distance")
print("   ✓ marsabit_roi boundary")
print("\n✅ Notebook 2 ready — no satellite reprocessing needed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.3 MB/s eta 0:00:00
✅ GEE authenticated and initialized!

📦 Assets loaded from: projects/mironga-project-marsabit/assets/marsabit/
   ✓ ndvi_1990, ndvi_2000, ndvi_2010, ndvi_2020, ndvi_2024
   ✓ ndvi_change layers (4 decadal + 1 total)
   ✓ rainfall_change, elevation, lci_distance
   ✓ marsabit_roi boundary

✅ Notebook 2 ready — no satellite reprocessing needed!


## 🗺️ STEP 2 — Load Marsabit Boundary & Build Feature Stack

**What we are doing:**
Before we can classify land cover, we need to build a "feature stack" — a
multi-band image where each band is a different variable that helps the
Random Forest model tell land cover classes apart.

**Why one band (NDVI alone) is not enough:**
NDVI tells us how green a pixel is — but it cannot distinguish between:
- A wetland and a river (both have high NDVI near water)
- Rangeland and degraded forest (similar NDVI values)

By stacking multiple bands, we give the classifier more information to
work with — making it smarter and more accurate.

**The feature bands we stack for each time period:**
| Band | What it measures | Why it helps |
|------|-----------------|--------------|
| NDVI | Vegetation greenness | Primary vegetation signal |
| NIR | Near-infrared reflectance | Separates vegetation types |
| Red | Red reflectance | Separates soil from vegetation |
| SWIR | Shortwave infrared | Separates wetlands from dryland |
| Elevation | Terrain height | Forest only exists at high elevation |
| Slope | Terrain steepness | Affects grazing accessibility |

**Output of this step:**
A feature stack image for each of the 5 time periods — each with 6 bands.
The Random Forest classifier will train and predict on these stacks.

In [5]:
# ─────────────────────────────────────────────────────────────────────
# We need the original Landsat composites to extract NIR, Red, SWIR
# These raw band values give the classifier more to work with than
# NDVI alone. We reload them from the original GEE collections.
# ─────────────────────────────────────────────────────────────────────

def apply_scale_factors(image):
    """Apply Landsat C2 L2 scale factors to optical bands."""
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True)

def mask_landsat_clouds(image):
    """Mask clouds and cloud shadows using QA_PIXEL band."""
    qa = image.select('QA_PIXEL')
    cloud_shadow = qa.bitwiseAnd(1 << 3).eq(0)
    cloud        = qa.bitwiseAnd(1 << 5).eq(0)
    return image.updateMask(cloud_shadow.And(cloud))

def build_feature_stack(collection_id, start_year, end_year,
                        roi, sensor='L57'):
    """
    Builds a multi-band feature stack for classification.
    Bands: NDVI, NIR, Red, SWIR, Elevation, Slope

    Args:
        collection_id : GEE Landsat collection string
        start_year    : Start year as string
        end_year      : End year as string
        roi           : GEE geometry
        sensor        : 'L57' for Landsat 5/7 | 'L89' for Landsat 8/9

    Returns:
        6-band image clipped to roi
    """
    collection = (
        ee.ImageCollection(collection_id)
        .filterBounds(roi)
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
        .map(mask_landsat_clouds)
        .map(apply_scale_factors)
    )

    composite = collection.median().clip(roi)

    # Select bands based on sensor generation
    if sensor == 'L57':
        nir_b, red_b, swir_b = 'SR_B4', 'SR_B3', 'SR_B5'
    else:
        nir_b, red_b, swir_b = 'SR_B5', 'SR_B4', 'SR_B6'

    nir  = composite.select(nir_b).rename('NIR')
    red  = composite.select(red_b).rename('Red')
    swir = composite.select(swir_b).rename('SWIR')
    ndvi = composite.normalizedDifference([nir_b, red_b]).rename('NDVI')

    # Elevation and slope (same for all time periods)
    srtm  = ee.Image("USGS/SRTMGL1_003")
    elev  = srtm.select('elevation').rename('Elevation')
    slope = ee.Terrain.slope(srtm).rename('Slope')

    # Stack all bands into one image
    stack = (ndvi
             .addBands(nir)
             .addBands(red)
             .addBands(swir)
             .addBands(elev)
             .addBands(slope)
             .clip(roi))

    return stack


# ─────────────────────────────────────────────────────────────────────
# Build feature stacks for all 5 time periods
# ─────────────────────────────────────────────────────────────────────
print("⏳ Building feature stacks for all 5 periods...")

stack_1990 = build_feature_stack(
    'LANDSAT/LT05/C02/T1_L2', '1988', '1992', marsabit_geom, 'L57')
print("   ✓ Feature stack 1990 (bands: NDVI, NIR, Red, SWIR, Elevation, Slope)")

stack_2000 = build_feature_stack(
    'LANDSAT/LT05/C02/T1_L2', '1998', '2002', marsabit_geom, 'L57')
print("   ✓ Feature stack 2000")

stack_2010 = build_feature_stack(
    'LANDSAT/LE07/C02/T1_L2', '2008', '2012', marsabit_geom, 'L57')
print("   ✓ Feature stack 2010")

stack_2020 = build_feature_stack(
    'LANDSAT/LC08/C02/T1_L2', '2018', '2022', marsabit_geom, 'L89')
print("   ✓ Feature stack 2020")

stack_2024 = build_feature_stack(
    'LANDSAT/LC09/C02/T1_L2', '2022', '2024', marsabit_geom, 'L89')
print("   ✓ Feature stack 2024")

# Visualize the 2020 stack as a true colour composite to confirm
rgb_vis = {'bands': ['NIR', 'Red', 'SWIR'], 'min': 0, 'max': 0.4}

Map = geemap.Map()
Map.setCenter(37.9, 2.3, 8)
Map.addLayer(stack_2020, rgb_vis, 'False Colour Composite 2020')
Map.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map.addLayerControl()

print("\n✅ All 5 feature stacks ready!")
print("   Each stack has 6 bands: NDVI, NIR, Red, SWIR, Elevation, Slope")
print("   Map shows 2020 false colour composite — vegetation appears red/pink")
Map

⏳ Building feature stacks for all 5 periods...
   ✓ Feature stack 1990 (bands: NDVI, NIR, Red, SWIR, Elevation, Slope)
   ✓ Feature stack 2000
   ✓ Feature stack 2010
   ✓ Feature stack 2020
   ✓ Feature stack 2024

✅ All 5 feature stacks ready!
   Each stack has 6 bands: NDVI, NIR, Red, SWIR, Elevation, Slope
   Map shows 2020 false colour composite — vegetation appears red/pink


Map(center=[2.3, 37.9], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(c…